# Capítulos 7-10: Funções, Métodos Numéricos e Projetos Integrados

**Disciplina:** FSC1189 - Algoritmo e Programação

Exercícios avançados com métodos numéricos e simulações físicas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import fsolve

## Exercício 7.1: Módulo de Funções Físicas

**Enunciado:**

Crie um módulo com funções para calcular energias em diferentes contextos.

In [ ]:
class FisicaBasica:
    """Módulo com funções básicas de Física."""
    
    g = 9.81  # aceleração da gravidade (m/s²)
    
    @staticmethod
    def energia_cinetica(massa, velocidade):
        """
        Calcula energia cinética: Ec = (1/2)mv²
        
        Args:
            massa: em kg
            velocidade: em m/s
        
        Returns:
            float: energia em joules
        """
        return 0.5 * massa * velocidade**2
    
    @staticmethod
    def energia_potencial(massa, altura, g=9.81):
        """
        Calcula energia potencial gravitacional: Ep = mgh
        
        Args:
            massa: em kg
            altura: em metros
            g: aceleração da gravidade (padrão 9.81 m/s²)
        
        Returns:
            float: energia em joules
        """
        return massa * g * altura
    
    @staticmethod
    def energia_mecanica(massa, velocidade, altura, g=9.81):
        """
        Calcula energia mecânica total: Em = Ec + Ep
        """
        Ec = FisicaBasica.energia_cinetica(massa, velocidade)
        Ep = FisicaBasica.energia_potencial(massa, altura, g)
        return Ec + Ep
    
    @staticmethod
    def trabalho(forca, distancia, cosseno_angulo=1):
        """
        Calcula trabalho: W = F·d·cos(θ)
        
        Args:
            forca: força em newtons
            distancia: em metros
            cosseno_angulo: cos(θ), onde θ é ângulo entre F e d
        
        Returns:
            float: trabalho em joules
        """
        return forca * distancia * cosseno_angulo

# Teste
m = 5.0  # kg
v = 10.0  # m/s
h = 2.0  # m

Ec = FisicaBasica.energia_cinetica(m, v)
Ep = FisicaBasica.energia_potencial(m, h)
Em = FisicaBasica.energia_mecanica(m, v, h)

print("=== ENERGIAS DE UM OBJETO ===")
print(f"Massa: {m} kg")
print(f"Velocidade: {v} m/s")
print(f"Altura: {h} m")
print()
print(f"Energia cinética: {Ec:.2f} J")
print(f"Energia potencial: {Ep:.2f} J")
print(f"Energia mecânica: {Em:.2f} J")

---

## Exercício 8.1: Método da Bisseção

**Enunciado:**

Encontre a raiz da função $f(x) = x^3 - 2$ no intervalo [1, 2] usando o método da bisseção.

In [ ]:
def f_exemplo(x):
    """Função de teste: f(x) = x³ - 2"""
    return x**3 - 2

def metodo_bisseccao(f, a, b, tolerancia=1e-6, max_iter=100):
    """
    Encontra raiz usando o método da bisseção.
    
    Args:
        f: função
        a, b: intervalo [a, b]
        tolerancia: precisão desejada
        max_iter: máximo de iterações
    
    Returns:
        tuple: (raiz, iteracoes, erros)
    """
    if f(a) * f(b) >= 0:
        raise ValueError("f(a) e f(b) devem ter sinais opostos")
    
    erros = []
    
    for i in range(max_iter):
        c = (a + b) / 2
        fc = f(c)
        erro = abs(fc)
        erros.append(erro)
        
        if erro < tolerancia:
            return c, i + 1, erros
        
        if f(a) * fc < 0:
            b = c
        else:
            a = c
    
    return c, max_iter, erros

# Resolver
raiz, iteracoes, erros = metodo_bisseccao(f_exemplo, 1, 2)
raiz_real = 2**(1/3)

print("=== MÉTODO DA BISSEÇÃO ===")
print(f"Função: f(x) = x³ - 2")
print(f"Intervalo: [1, 2]")
print()
print(f"Raiz encontrada: {raiz:.10f}")
print(f"Raiz real: {raiz_real:.10f}")
print(f"Diferença: {abs(raiz - raiz_real):.2e}")
print(f"Número de iterações: {iteracoes}")
print(f"f(raiz) = {f_exemplo(raiz):.2e}")

### Convergência do Método

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico da função
x = np.linspace(0.5, 2.5, 200)
y = f_exemplo(x)
ax1.plot(x, y, 'b-', linewidth=2, label='f(x) = x³ - 2')
ax1.axhline(0, color='k', linestyle='-', linewidth=0.5)
ax1.plot(raiz, 0, 'ro', markersize=10, label=f'Raiz = {raiz:.6f}')
ax1.set_xlabel('x', fontsize=11)
ax1.set_ylabel('f(x)', fontsize=11)
ax1.set_title('Função e Raiz Encontrada', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Convergência dos erros
iteracoes_plot = list(range(1, len(erros) + 1))
ax2.semilogy(iteracoes_plot, erros, 'g-o', linewidth=2, markersize=4)
ax2.set_xlabel('Iteração', fontsize=11)
ax2.set_ylabel('|f(c)| (escala log)', fontsize=11)
ax2.set_title('Convergência do Método', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## Exercício 8.2: Diferenciação Numérica

**Enunciado:**

Calcule numericamente a derivada de $f(x) = \sin(x)$ usando diferenças finitas e compare com o resultado analítico (que é $\cos(x)$).

In [ ]:
def f_sen(x):
    return np.sin(x)

def f_sen_derivada_real(x):
    return np.cos(x)

def derivada_numerica_central(f, x, h=1e-5):
    """
    Calcula derivada usando diferença central:
    f'(x) ≈ [f(x+h) - f(x-h)] / (2h)
    """
    return (f(x + h) - f(x - h)) / (2 * h)

def derivada_numerica_progressiva(f, x, h=1e-5):
    """
    Calcula derivada usando diferença progressiva:
    f'(x) ≈ [f(x+h) - f(x)] / h
    """
    return (f(x + h) - f(x)) / h

# Teste para x = π/4
x_teste = np.pi / 4

der_real = f_sen_derivada_real(x_teste)
der_central = derivada_numerica_central(f_sen, x_teste)
der_progressiva = derivada_numerica_progressiva(f_sen, x_teste)

print("=== DIFERENCIAÇÃO NUMÉRICA ===")
print(f"Função: f(x) = sin(x)")
print(f"Ponto: x = π/4 ≈ {x_teste:.6f}")
print()
print(f"Derivada real (cos(π/4)): {der_real:.10f}")
print(f"Derivada numérica (diferença central): {der_central:.10f}")
print(f"Derivada numérica (diferença progressiva): {der_progressiva:.10f}")
print()
print(f"Erro (central): {abs(der_central - der_real):.2e}")
print(f"Erro (progressiva): {abs(der_progressiva - der_real):.2e}")

### Análise de Precisão

In [ ]:
# Variar h e analisar erro
h_valores = np.logspace(-8, -2, 50)
erros_central = []
erros_progressiva = []

for h in h_valores:
    der_c = derivada_numerica_central(f_sen, x_teste, h)
    der_p = derivada_numerica_progressiva(f_sen, x_teste, h)
    
    erros_central.append(abs(der_c - der_real))
    erros_progressiva.append(abs(der_p - der_real))

plt.figure(figsize=(10, 6))
plt.loglog(h_valores, erros_central, 'b-o', label='Diferença Central', markersize=4)
plt.loglog(h_valores, erros_progressiva, 'r-s', label='Diferença Progressiva', markersize=4)
plt.xlabel('Tamanho do passo (h)', fontsize=11)
plt.ylabel('Erro Absoluto (escala log)', fontsize=11)
plt.title('Análise de Precisão - Diferenciação Numérica', fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

---

## Exercício 8.3: Integração Numérica - Regra do Trapézio

**Enunciado:**

Calcule a integral $\int_0^{\pi} \sin(x) dx$ usando a regra do trapézio.

Valor teórico: $\int_0^{\pi} \sin(x) dx = 2$

In [ ]:
def regra_trapezoidal(f, a, b, n):
    """
    Calcula integral usando regra dos trapézios.
    ∫_a^b f(x)dx ≈ (h/2)[f(x₀) + 2f(x₁) + 2f(x₂) + ... + f(xₙ)]
    """
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    
    # Fórmula dos trapézios
    integral = h * (y[0] + 2*np.sum(y[1:-1]) + y[-1]) / 2
    
    return integral, x, y

# Calcular para diferentes números de subintervalos
a = 0
b = np.pi
valor_teorico = 2.0

n_valores = [10, 50, 100, 500, 1000]

print("=== INTEGRAÇÃO NUMÉRICA - REGRA DO TRAPÉZIO ===")
print(f"Integral: ∫₀^π sin(x)dx")
print(f"Valor teórico: {valor_teorico}")
print()
print("n     | Valor Aproximado | Erro Absoluto | Erro (%)")
print("-" * 55)

erros = []
for n in n_valores:
    integral, _, _ = regra_trapezoidal(np.sin, a, b, n)
    erro = abs(integral - valor_teorico)
    erro_percent = (erro / valor_teorico) * 100
    
    erros.append(erro)
    print(f"{n:<5} | {integral:16.10f} | {erro:13.2e} | {erro_percent:7.4f}%")

### Visualização

In [ ]:
# Plotar aproximação com n=20
n = 20
integral, x_trap, y_trap = regra_trapezoidal(np.sin, 0, np.pi, n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico da aproximação
x_continuo = np.linspace(0, np.pi, 200)
y_continuo = np.sin(x_continuo)

ax1.plot(x_continuo, y_continuo, 'b-', linewidth=2, label='sin(x)')
ax1.fill_between(x_continuo, y_continuo, alpha=0.2, color='blue')
ax1.plot(x_trap, y_trap, 'ro-', markersize=5, label=f'Aproximação (n={n})')

# Trapézios
for i in range(len(x_trap) - 1):
    ax1.fill_between([x_trap[i], x_trap[i+1]], 
                     [y_trap[i], y_trap[i+1]], 
                     alpha=0.1, color='red')

ax1.set_xlabel('x', fontsize=11)
ax1.set_ylabel('sin(x)', fontsize=11)
ax1.set_title(f'Aproximação com Trapézios (n={n})', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Convergência do erro
ax2.loglog(n_valores, erros, 'go-', linewidth=2, markersize=8)
ax2.set_xlabel('Número de subintervalos (n)', fontsize=11)
ax2.set_ylabel('Erro Absoluto (escala log)', fontsize=11)
ax2.set_title('Convergência da Regra do Trapézio', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## Exercício 8.4: Integração Numérica - Regra de Simpson

**Enunciado:**

Calcule a integral $\int_0^{\pi} \sin(x) dx$ usando a regra de Simpson (1/3).

A regra de Simpson tem precisão de ordem O(h⁴), melhor que o trapézio O(h²).

### Visualização e Comparação de Métodos

In [ ]:
# Gráfico de convergência de erros
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Recalcular erros para o gráfico
n_plot = np.array([5, 10, 20, 50, 100, 200, 500, 1000])
erros_trap_plot = []
erros_simp_plot = []

for n in n_plot:
    integral_trap, _, _ = regra_trapezoidal(np.sin, a, b, n)
    integral_simp, _, _ = regra_simpson(np.sin, a, b, n)
    
    erros_trap_plot.append(abs(integral_trap - valor_teorico))
    erros_simp_plot.append(abs(integral_simp - valor_teorico))

# Painel 1: Log-log plot
ax1.loglog(n_plot, erros_trap_plot, 'b-o', linewidth=2, markersize=8, label='Trapézio (O(h²))')
ax1.loglog(n_plot, erros_simp_plot, 'r-s', linewidth=2, markersize=8, label='Simpson (O(h⁴))')

# Linhas de referência
n_ref = np.array([5, 1000])
# O(h²) ∝ 1/n²
h = (b - a) / n_ref
ref_h2 = 0.001 / n_ref**2
ax1.loglog(n_ref, ref_h2, 'b--', alpha=0.5, linewidth=1, label='Referência O(1/n²)')

# O(h⁴) ∝ 1/n⁴
ref_h4 = 0.0001 / n_ref**4
ax1.loglog(n_ref, ref_h4, 'r--', alpha=0.5, linewidth=1, label='Referência O(1/n⁴)')

ax1.set_xlabel('Número de subintervalos (n)', fontsize=11)
ax1.set_ylabel('Erro Absoluto (escala log)', fontsize=11)
ax1.set_title('Convergência: Comparação Trapézio × Simpson', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, which='both')

# Painel 2: Visualização das fórmulas
x_simp = np.linspace(a, b, 9)
y_simp = np.sin(x_simp)

x_continuo = np.linspace(a, b, 200)
y_continuo = np.sin(x_continuo)

ax2.plot(x_continuo, y_continuo, 'k-', linewidth=2.5, label='sin(x)')
ax2.fill_between(x_continuo, y_continuo, alpha=0.15, color='blue')

# Desenhar parábolas para Simpson (usando polinômios por pares de intervalos)
for i in range(0, len(x_simp) - 2, 2):
    x_parab = np.linspace(x_simp[i], x_simp[i+2], 30)
    # Interpolação quadrática
    coeffs = np.polyfit([x_simp[i], x_simp[i+1], x_simp[i+2]], 
                        [y_simp[i], y_simp[i+1], y_simp[i+2]], 2)
    y_parab = np.polyval(coeffs, x_parab)
    ax2.fill_between(x_parab, y_parab, alpha=0.2, color='red')
    ax2.plot(x_parab, y_parab, 'r-', linewidth=1.5)

ax2.plot(x_simp, y_simp, 'ro', markersize=8, label='Pontos de avaliação')
ax2.set_xlabel('x', fontsize=11)
ax2.set_ylabel('sin(x)', fontsize=11)
ax2.set_title('Regra de Simpson (n=8): Parábolas vs Função Real', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(a, b)

plt.tight_layout()
plt.show()

In [ ]:
def regra_simpson(f, a, b, n):
    """
    Calcula integral usando regra de Simpson (1/3).
    
    Fórmula: ∫_a^b f(x)dx ≈ (h/3)[f(x₀) + 4f(x₁) + 2f(x₂) + 4f(x₃) + ... + f(xₙ)]
    
    Nota: n deve ser par para aplicar corretamente
    
    Args:
        f: função a integrar
        a, b: limites de integração
        n: número de subintervalos (deve ser par)
    
    Returns:
        integral: valor aproximado da integral
        x: pontos de avaliação
        y: valores da função nos pontos
    """
    if n % 2 != 0:
        n += 1  # Garantir que n seja par
    
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    
    # Coeficientes de Simpson
    coeficientes = np.ones(n + 1)
    coeficientes[0] = 1
    coeficientes[-1] = 1
    coeficientes[1:-1:2] = 4  # índices ímpares
    coeficientes[2:-1:2] = 2  # índices pares (exceto primeiro e último)
    
    integral = (h / 3) * np.sum(coeficientes * y)
    
    return integral, x, y

# Calcular para diferentes números de subintervalos
a = 0
b = np.pi
valor_teorico = 2.0

n_valores = [10, 50, 100, 500, 1000]

print("=== INTEGRAÇÃO NUMÉRICA - REGRA DE SIMPSON ===")
print(f"Integral: ∫₀^π sin(x)dx")
print(f"Valor teórico: {valor_teorico}")
print()
print("n     | Valor Aproximado | Erro Absoluto | Erro (%)")
print("-" * 55)

erros_simpson = []
for n in n_valores:
    integral, _, _ = regra_simpson(np.sin, a, b, n)
    erro = abs(integral - valor_teorico)
    erro_percent = (erro / valor_teorico) * 100
    
    erros_simpson.append(erro)
    print(f"{n:<5} | {integral:16.10f} | {erro:13.2e} | {erro_percent:7.4f}%")

# Comparar com trapézio
print("\n=== COMPARAÇÃO: SIMPSON vs TRAPÉZIO ===")
print()
print("n     | Erro Trapézio | Erro Simpson | Melhoria")
print("-" * 50)

for i, n in enumerate(n_valores):
    integral_trap, _, _ = regra_trapezoidal(np.sin, a, b, n)
    integral_simp, _, _ = regra_simpson(np.sin, a, b, n)
    
    erro_trap = abs(integral_trap - valor_teorico)
    erro_simp = abs(integral_simp - valor_teorico)
    
    if erro_trap > 0:
        melhoria = erro_trap / erro_simp
    else:
        melhoria = float('inf')
    
    print(f"{n:<5} | {erro_trap:13.2e} | {erro_simp:12.2e} | {melhoria:7.2f}x")

---

## Exercício 10.1: Projeto - Análise de Dados de Pêndulo Simples

**Cenário:**

Dados experimentais de período de um pêndulo em função do comprimento.
Objetivo: Estimar $g$ a partir da relação $T = 2\pi\sqrt{L/g}$

In [ ]:
# Dados experimentais (simulados com ruído)
np.random.seed(42)
L_teorico = np.array([0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0])  # metros
g_real = 9.81  # m/s²

# Calcular períodos teóricos
T_teorico = 2 * np.pi * np.sqrt(L_teorico / g_real)

# Adicionar ruído experimental (1%)
ruido = 0.01 * T_teorico * np.random.randn(len(T_teorico))
T_medido = T_teorico + ruido

# Criar tabela
df_pendulo = pd.DataFrame({
    'Comprimento L (m)': L_teorico,
    'Período T teórico (s)': T_teorico,
    'Período T medido (s)': T_medido,
    'Erro (%)': (np.abs(T_medido - T_teorico) / T_teorico) * 100
})

print("=== DADOS DE PÊNDULO SIMPLES ===")
print(df_pendulo.to_string(index=False))

### Análise de Dados

In [ ]:
# Usar relação: T² = (4π²/g)L para ajustar uma reta
# Se plotar T² vs L, a inclinação = 4π²/g

T_squared = T_medido**2

# Regressão linear
# T² = aL + b, onde a = 4π²/g
coef = np.polyfit(L_teorico, T_squared, 1)
a = coef[0]
b = coef[1]

g_estimado = 4 * np.pi**2 / a

print("\n=== ANÁLISE DE REGRESSÃO LINEAR ===")
print(f"Ajuste: T² = {a:.6f}·L + {b:.6f}")
print(f"Coeficiente angular (a): {a:.6f}")
print(f"Coeficiente linear (b): {b:.6f}")
print()
print(f"g estimado: {g_estimado:.3f} m/s²")
print(f"g real: {g_real:.3f} m/s²")
print(f"Erro: {abs(g_estimado - g_real):.4f} m/s² ({(abs(g_estimado - g_real)/g_real)*100:.2f}%)")

### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico T vs L
ax1.scatter(L_teorico, T_medido, s=80, alpha=0.7, label='Dados medidos', color='blue')
ax1.plot(L_teorico, T_teorico, 'r--', linewidth=2, label='Valor teórico')
ax1.set_xlabel('Comprimento L (m)', fontsize=11)
ax1.set_ylabel('Período T (s)', fontsize=11)
ax1.set_title('Período vs Comprimento', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Gráfico T² vs L (regressão linear)
L_fit = np.linspace(L_teorico.min(), L_teorico.max(), 100)
T2_fit = a * L_fit + b

ax2.scatter(L_teorico, T_squared, s=80, alpha=0.7, label='Dados medidos', color='green')
ax2.plot(L_fit, T2_fit, 'b-', linewidth=2, label=f'Ajuste linear')
ax2.set_xlabel('Comprimento L (m)', fontsize=11)
ax2.set_ylabel('Período² T² (s²)', fontsize=11)
ax2.set_title('T² vs L - Regressão Linear', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercício 10.2: Projeto - Simulação de Queda Livre com Arrasto

**Cenário:**

Simular a queda de um objeto considerando resistência do ar.

In [ ]:
def queda_livre_com_arrasto(condicoes_iniciais, parametros, t_total):
    """
    Simula queda livre com arrasto quadrático.
    
    dv/dt = g - (b/m)v|v|
    """
    def derivada(y, t, g, b, m):
        v = y[0]
        aceleracao = g - (b / m) * v * abs(v)
        return [aceleracao]
    
    g = parametros['g']
    b = parametros['b']  # coeficiente de arrasto
    m = parametros['m']  # massa
    
    t = np.linspace(0, t_total, 500)
    solucao = odeint(derivada, condicoes_iniciais, t, args=(g, b, m))
    
    return t, solucao[:, 0]

# Parâmetros
parametros = {
    'g': 9.81,      # aceleração da gravidade
    'b': 0.1,       # coeficiente de arrasto
    'm': 1.0        # massa em kg
}

# Velocidade terminal
v_terminal = np.sqrt(parametros['g'] * parametros['m'] / parametros['b'])

# Simular
t, v = queda_livre_com_arrasto([0], parametros, 10)

# Também simular sem arrasto para comparação
parametros_sem_arrasto = parametros.copy()
parametros_sem_arrasto['b'] = 0.001  # arrasto negligenciável
t2, v_sem_arrasto = queda_livre_com_arrasto([0], parametros_sem_arrasto, 10)

print("=== SIMULAÇÃO: QUEDA COM ARRASTO ===")
print(f"Massa: {parametros['m']} kg")
print(f"Coeficiente de arrasto: {parametros['b']}")
print(f"Velocidade terminal: {v_terminal:.2f} m/s")

### Visualização

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(t, v, 'b-', linewidth=2, label=f'Com arrasto (b={parametros["b"]})')
plt.plot(t2, v_sem_arrasto, 'r--', linewidth=2, label='Sem arrasto')
plt.axhline(v_terminal, color='g', linestyle=':', linewidth=2, label=f'Velocidade terminal = {v_terminal:.2f} m/s')

plt.xlabel('Tempo (s)', fontsize=11)
plt.ylabel('Velocidade (m/s)', fontsize=11)
plt.title('Queda Livre: Com e Sem Arrasto', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Exercício Proposto 5: Lei de Coulomb Numérica

**Desafio:**

Implemente o cálculo da força eletrostática entre múltiplas cargas em um espaço 2D.

$$F = k \frac{|q_1 q_2|}{r^2}$$

Visualize as forças com vetores.

In [ ]:
k = 8.99e9  # constante de Coulomb

def distancia(p1, p2):
    """Calcula a distância entre dois pontos."""
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def campo_eletrico(x, y, cargas):
    """
    Calcula o campo elétrico em um ponto (x, y) devido a múltiplas cargas.
    
    Args:
        x, y: coordenadas do ponto
        cargas: lista de tuplas (x_carga, y_carga, q_carga)
    
    Returns:
        Ex, Ey: componentes do campo elétrico
    """
    Ex, Ey = 0, 0
    
    for x_c, y_c, q in cargas:
        r = distancia((x, y), (x_c, y_c))
        
        if r < 0.1:  # Evitar singularidade muito próxima
            continue
        
        # Campo elétrico = k*q/r² (magnitud)
        E = k * q / (r**2)
        
        # Componentes
        cos_theta = (x - x_c) / r
        sin_theta = (y - y_c) / r
        
        Ex += E * cos_theta
        Ey += E * sin_theta
    
    return Ex, Ey

def potencial_eletrico(x, y, cargas):
    """Calcula o potencial elétrico em um ponto."""
    V = 0
    for x_c, y_c, q in cargas:
        r = distancia((x, y), (x_c, y_c))
        if r < 0.05:
            continue
        V += k * q / r
    return V

# Sistema de 3 cargas
cargas = [
    (-1, 0, 1e-6),      # +1 µC
    (1, 0, 1e-6),       # +1 µC
    (0, 1.5, -2e-6),    # -2 µC
]

# Grade para cálculos
x = np.linspace(-3, 3, 40)
y = np.linspace(-3, 3, 40)
X, Y = np.meshgrid(x, y)

# Calcular campo elétrico
U = np.zeros_like(X)
V_pot = np.zeros_like(X)

for i in range(len(x)):
    for j in range(len(y)):
        U[i, j], V_pot[i, j] = campo_eletrico(X[i, j], Y[i, j], cargas)

# Calcular potencial
for i in range(len(x)):
    for j in range(len(y)):
        V_pot[i, j] = potencial_eletrico(X[i, j], Y[i, j], cargas)

# Normalizar campo para streamplot
E_magnitude = np.sqrt(U**2 + V_pot**2)
U_norm = np.log10(E_magnitude + 1) * np.sign(U)
V_norm = np.log10(E_magnitude + 1) * np.sign(V_pot)

# Visualização
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Painel 1: Linhas de campo com streamplot
strm = ax1.streamplot(X, Y, U_norm, V_norm, color=np.log10(E_magnitude + 1), 
                      cmap='hot', density=1.5, linewidth=1)
cbar1 = plt.colorbar(strm.lines, ax=ax1, label='log₁₀(|E|)')

# Marcar cargas
for x_c, y_c, q in cargas:
    cor = 'red' if q > 0 else 'blue'
    marcador = '+' if q > 0 else '_'
    ax1.scatter(x_c, y_c, s=300, c=cor, marker='o', edgecolors='black', linewidth=2, zorder=5)

ax1.set_xlabel('x (m)', fontsize=11)
ax1.set_ylabel('y (m)', fontsize=11)
ax1.set_title('Linhas de Campo Elétrico', fontsize=12, fontweight='bold')
ax1.set_xlim(-3, 3)
ax1.set_ylim(-3, 3)
ax1.grid(True, alpha=0.2)

# Painel 2: Equipotenciais com contourf
contourf = ax2.contourf(X, Y, V_pot, levels=20, cmap='viridis')
contour = ax2.contour(X, Y, V_pot, levels=10, colors='white', linewidths=0.5, alpha=0.5)
ax2.clabel(contour, inline=True, fontsize=8)
cbar2 = plt.colorbar(contourf, ax=ax2, label='Potencial V (V)')

# Marcar cargas
for x_c, y_c, q in cargas:
    cor = 'red' if q > 0 else 'blue'
    ax2.scatter(x_c, y_c, s=300, c=cor, marker='o', edgecolors='white', linewidth=2, zorder=5)

ax2.set_xlabel('x (m)', fontsize=11)
ax2.set_ylabel('y (m)', fontsize=11)
ax2.set_title('Equipotenciais (Linhas de Potencial Constante)', fontsize=12, fontweight='bold')
ax2.set_xlim(-3, 3)
ax2.set_ylim(-3, 3)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("=== CAMPO ELETROSTÁTICO 2D ===")
print("\nCargas no sistema:")
for i, (x_c, y_c, q) in enumerate(cargas, 1):
    print(f"Carga {i}: q = {q*1e6:.1f} µC em ({x_c}, {y_c})")

---

## Exercício Proposto 6: Oscilador Harmônico Amortecido

**Desafio:**

Simule o movimento de um oscilador harmônico com amortecimento:

$$\frac{d^2x}{dt^2} + 2\gamma\frac{dx}{dt} + \omega_0^2 x = 0$$

Teste para diferentes graus de amortecimento (subamortecido, criticamente amortecido, superamortecido).

In [ ]:
def oscilador_amortecido(condicoes_iniciais, omega0, gamma, t_total):
    """
    Simula oscilador harmônico amortecido.
    
    EDO: d²x/dt² + 2γ(dx/dt) + ω₀²x = 0
    
    Args:
        condicoes_iniciais: [x0, v0]
        omega0: frequência natural
        gamma: coeficiente de amortecimento
        t_total: tempo total de simulação
    
    Returns:
        t, x, v: tempo, deslocamento, velocidade
    """
    def derivadas(y, t, omega0, gamma):
        x, v = y
        dx_dt = v
        dv_dt = -2 * gamma * v - omega0**2 * x
        return [dx_dt, dv_dt]
    
    t = np.linspace(0, t_total, 1000)
    solucao = odeint(derivadas, condicoes_iniciais, t, args=(omega0, gamma))
    
    return t, solucao[:, 0], solucao[:, 1]

# Parâmetros
omega0 = 2.0  # frequência natural (rad/s)
x0 = 1.0      # deslocamento inicial
v0 = 0        # velocidade inicial

# Três regimes de amortecimento
# Subamortecido: γ < ω₀
# Criticamente amortecido: γ = ω₀
# Superamortecido: γ > ω₀

regimes = [
    (0.5, "Subamortecido (γ=0.5, γ<ω₀)"),
    (2.0, "Criticamente amortecido (γ=2.0, γ=ω₀)"),
    (3.0, "Superamortecido (γ=3.0, γ>ω₀)"),
]

# Simulações
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for idx, (gamma, titulo) in enumerate(regimes):
    t, x, v = oscilador_amortecido([x0, v0], omega0, gamma, 10)
    
    # Painel superior: deslocamento
    ax = axes[0, idx]
    ax.plot(t, x, 'b-', linewidth=2, label='x(t)')
    
    # Envelope exponencial para subamortecido
    if gamma < omega0:
        envelope = x0 * np.exp(-gamma * t)
        ax.plot(t, envelope, 'r--', linewidth=1, alpha=0.7, label='Envelope: exp(-γt)')
        ax.plot(t, -envelope, 'r--', linewidth=1, alpha=0.7)
    
    ax.axhline(0, color='k', linewidth=0.5, alpha=0.5)
    ax.set_ylabel('Deslocamento x (m)', fontsize=10)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    if idx == 0:
        ax.set_ylabel('Deslocamento x (m)', fontsize=10, fontweight='bold')
    
    # Painel inferior: retrato de fase (x vs v)
    ax = axes[1, idx]
    ax.plot(x, v, 'g-', linewidth=2)
    ax.scatter(x0, v0, s=100, c='red', marker='o', zorder=5, label='Inicial')
    ax.scatter(0, 0, s=100, c='black', marker='x', zorder=5, label='Equilíbrio')
    
    # Adicionar setas para mostrar direção
    for i in range(0, len(t), 50):
        if i + 1 < len(t):
            ax.annotate('', xy=(x[i+1], v[i+1]), xytext=(x[i], v[i]),
                       arrowprops=dict(arrowstyle='->', color='green', alpha=0.5))
    
    ax.set_xlabel('Deslocamento x (m)', fontsize=10)
    ax.set_ylabel('Velocidade v (m/s)', fontsize=10)
    ax.set_title(f'Retrato de Fase - {titulo.split("(")[0].strip()}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    if idx == 0:
        ax.set_ylabel('Velocidade v (m/s)', fontsize=10, fontweight='bold')

plt.suptitle('Oscilador Harmônico Amortecido - Três Regimes de Amortecimento', 
             fontsize=13, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("=== OSCILADOR HARMÔNICO AMORTECIDO ===")
print(f"\nParâmetros:")
print(f"  Frequência natural: ω₀ = {omega0} rad/s")
print(f"  Deslocamento inicial: x₀ = {x0} m")
print(f"  Velocidade inicial: v₀ = {v0} m/s")
print(f"\nRegimes:")
print(f"  Subamortecido (γ < ω₀): oscila com amplitude decrescente")
print(f"  Criticamente amortecido (γ = ω₀): retorna ao equilíbrio mais rápido sem oscilar")
print(f"  Superamortecido (γ > ω₀): retorna lentamente sem oscilar")

---

## Análise Big-O: Complexidade Computacional

**Tema:** Medir empiricamente a complexidade de tempo de algoritmos de busca.

Comparar busca linear O(n) vs busca binária O(log n).

In [ ]:
import time
import bisect

def busca_linear(lista, alvo):
    \"\"\"Busca linear: O(n) - verifica cada elemento.\"\"\"\n    for i, elemento in enumerate(lista):\n        if elemento == alvo:\n            return i\n    return -1\n\ndef busca_binaria(lista, alvo):\n    \"\"\"Busca binária: O(log n) - divide e conquista.\"\"\"\n    esquerda, direita = 0, len(lista) - 1\n    \n    while esquerda <= direita:\n        meio = (esquerda + direita) // 2\n        if lista[meio] == alvo:\n            return meio\n        elif lista[meio] < alvo:\n            esquerda = meio + 1\n        else:\n            direita = meio - 1\n    \n    return -1\n\n# Teste com diferentes tamanhos\ntamanhos = [1000, 5000, 10000, 50000, 100000, 500000, 1000000]\ntempos_linear = []\ntempos_binaria = []\n\nprint(\"=== ANÁLISE DE COMPLEXIDADE (BIG-O) ===\")\nprint(f\"{'Tamanho':<10} | {'Linear (ms)':<15} | {'Binária (ms)':<15} | {'Razão':<10}\")\nprint(\"-\" * 55)\n\nfor tamanho in tamanhos:\n    # Criar lista ordenada\n    lista = list(range(tamanho))\n    alvo = tamanho // 2  # Buscar elemento no meio\n    \n    # Medir tempo - busca linear\n    num_tentativas = 100 if tamanho <= 10000 else 10\n    inicio = time.perf_counter()\n    for _ in range(num_tentativas):\n        busca_linear(lista, alvo)\n    tempo_linear = (time.perf_counter() - inicio) / num_tentativas * 1000\n    \n    # Medir tempo - busca binária\n    inicio = time.perf_counter()\n    for _ in range(num_tentativas):\n        busca_binaria(lista, alvo)\n    tempo_binaria = (time.perf_counter() - inicio) / num_tentativas * 1000\n    \n    tempos_linear.append(tempo_linear)\n    tempos_binaria.append(tempo_binaria)\n    \n    razao = tempo_linear / tempo_binaria if tempo_binaria > 0 else float('inf')\n    print(f\"{tamanho:<10} | {tempo_linear:<15.4f} | {tempo_binaria:<15.4f} | {razao:<10.1f}x\")\n\n# Visualização\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Painel 1: Escala linear\nax = axes[0]\nax.plot(tamanhos, tempos_linear, 'b-o', linewidth=2.5, markersize=8, label='Busca Linear (O(n))')\nax.plot(tamanhos, tempos_binaria, 'r-s', linewidth=2.5, markersize=8, label='Busca Binária (O(log n))')\nax.set_xlabel('Tamanho da lista (n)', fontsize=11, fontweight='bold')\nax.set_ylabel('Tempo (ms)', fontsize=11, fontweight='bold')\nax.set_title('Complexidade: Escala Linear', fontsize=12, fontweight='bold')\nax.legend(fontsize=11)\nax.grid(True, alpha=0.3)\n\n# Painel 2: Escala logarítmica (log-log)\nax = axes[1]\nax.loglog(tamanhos, tempos_linear, 'b-o', linewidth=2.5, markersize=8, label='Linear: O(n)')\nax.loglog(tamanhos, tempos_binaria, 'r-s', linewidth=2.5, markersize=8, label='Binária: O(log n)')\n\n# Linhas de referência\nn_ref = np.array([1000, 1000000])\nax.loglog(n_ref, 0.0001 * n_ref, 'b--', alpha=0.4, linewidth=1.5, label='Referência: ~n')\nax.loglog(n_ref, 0.1 * np.log2(n_ref), 'r--', alpha=0.4, linewidth=1.5, label='Referência: ~log₂(n)')\n\nax.set_xlabel('Tamanho da lista (n) - escala log', fontsize=11, fontweight='bold')\nax.set_ylabel('Tempo (ms) - escala log', fontsize=11, fontweight='bold')\nax.set_title('Complexidade: Escala Log-Log', fontsize=12, fontweight='bold')\nax.legend(fontsize=10)\nax.grid(True, alpha=0.3, which='both')\n\nplt.tight_layout()\nplt.show()\n\nprint(\"\\n=== INTERPRETAÇÃO ===\")\nprint(\"\\nBusca Linear (O(n)):\")\nprint(\"  • Tempo cresce linearmente com o tamanho\")\nprint(\"  • Cada duplicação do tamanho duplica o tempo\")\nprint(\"  • Adequada para listas pequenas\")\nprint(\"\\nBusca Binária (O(log n)):\")\nprint(\"  • Tempo cresce logaritmicamente\")\nprint(\"  • Aumentar 1.000.000x o tamanho só quadruplica o tempo!\")\nprint(\"  • Requer lista ordenada\")\nprint(\"  • Muito mais eficiente para listas grandes\")"